In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
def newest(name):
    hits = list(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(f'Missing {name}; attach the successful Day 9.16 notebook output.')
    return max(hits, key=lambda x: (x.stat().st_mtime, str(x)))

case_path = newest('day9_16_case_diagnostics.csv')
param_path = newest('day9_16_per_parameter_diagnostics.csv')
interval_path = newest('day9_16_conditional_intervals.csv')
cases, params, intervals = (pd.read_csv(x) for x in (case_path, param_path, interval_path))
need_case = {'simulation_row','status','max_r_hat','min_effective_sample_size','divergences','fraction_at_max_tree_depth'}
need_param = {'simulation_row','parameter','r_hat','effective_sample_size'}
if not need_case.issubset(cases): raise ValueError(f'Case CSV missing {sorted(need_case-set(cases.columns))}')
if not need_param.issubset(params): raise ValueError(f'Parameter CSV missing {sorted(need_param-set(params.columns))}')
for df in (cases, params, intervals): df['simulation_row'] = df['simulation_row'].astype(int)
if cases.simulation_row.duplicated().any() or set(cases.simulation_row) != {87483, 4923}:
    raise ValueError(f'Expected exactly rows 87483 and 4923; found {sorted(cases.simulation_row.tolist())}')
R_HAT_LIMIT, ESS_LIMIT = 1.05, 100

rows = []
for _, r in cases.iterrows():
    rid = int(r.simulation_row)
    decision = ('unresolved_sampler_geometry' if rid == 87483 else 'needs_review_low_noise_ESS')
    action = ('Keep unresolved; fixed-duration run remained non-mixing and all draws hit maximum tree depth.'
              if rid == 87483 else
              'Keep unresolved; log_noise ESS is below threshold and tree-depth saturation remains high.')
    rows.append({'simulation_row': rid, 'sampler_status': r.status, 'review_disposition': decision,
        'max_r_hat': r.max_r_hat, 'r_hat_limit': R_HAT_LIMIT,
        'min_effective_sample_size': r.min_effective_sample_size, 'ess_limit': ESS_LIMIT,
        'divergences': r.divergences, 'fraction_at_max_tree_depth': r.fraction_at_max_tree_depth,
        'period_fixed_days': r.get('period_fixed_days', np.nan), 'duration_fixed': r.get('duration_fixed', False),
        'next_action': action})
dispositions = pd.DataFrame(rows).sort_values('simulation_row')

def flag(r):
    rh, ess = float(r.r_hat), float(r.effective_sample_size)
    if not np.isfinite(rh) or not np.isfinite(ess): return 'diagnostic_missing_or_nonfinite'
    if rh > R_HAT_LIMIT and ess < ESS_LIMIT: return 'rhat_and_ess_fail'
    if rh > R_HAT_LIMIT: return 'rhat_fail'
    if ess < ESS_LIMIT: return 'ess_fail'
    return 'parameter_screen_pass'
params['parameter_flag'] = params.apply(flag, axis=1)
params = params.sort_values(['simulation_row','parameter'])

# Truth-in-interval counts are conditional synthetic checks, not calibration.
coverage_rows = []
if 'truth_inside_90pct_interval' in intervals:
    for rid, g in intervals.groupby('simulation_row'):
        if int(rid) == 87483 and 'parameter' in g:
            g = g[g.parameter.ne('duration_fraction')]  # duration was fixed, not inferred
        n = len(g)
        hits = int(g.truth_inside_90pct_interval.fillna(False).astype(bool).sum()) if n else 0
        coverage_rows.append({'simulation_row': int(rid), 'truth_inside_count': hits,
            'estimated_interval_count': n, 'scope': 'conditional synthetic check; not calibration'})
coverage = pd.DataFrame(coverage_rows)

OUT = WORK / ('day9-17-bayesian-review-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
dispositions.to_csv(OUT/'day9_17_case_dispositions.csv', index=False)
params.to_csv(OUT/'day9_17_parameter_flags.csv', index=False)
coverage.to_csv(OUT/'day9_17_conditional_truth_checks.csv', index=False)
n = len(dispositions)
passed = int(dispositions.sampler_status.eq('screen_pass').sum())
report = {'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_files': {'case_diagnostics': str(case_path), 'parameter_diagnostics': str(param_path), 'conditional_intervals': str(interval_path)},
    'cases_reviewed': n, 'screen_pass_cases': passed, 'needs_review_cases': n-passed,
    'thresholds': {'max_r_hat': R_HAT_LIMIT, 'min_ess': ESS_LIMIT, 'divergences': 0},
    'interpretation': ['Neither Day 9.16 targeted case passed all sampler thresholds.',
        'Row 87483 remains unresolved: fixing duration did not restore mixing; all draws hit maximum tree depth.',
        'Row 4923 remains under review: log_noise ESS is 53.4, below 100, with substantial tree-depth saturation.',
        'Zero divergences alone do not establish convergence.',
        'These are conditional synthetic checks, not calibration or evidence of real-star validity.'],
    'outputs': ['day9_17_case_dispositions.csv','day9_17_parameter_flags.csv',
        'day9_17_conditional_truth_checks.csv','day9_17_review_report.md']}
(OUT/'day9_17_review_summary.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
md = ['# Day 9.17 Bayesian sampler review', '',
    f'Cases reviewed: **{n}** | sampler screen passes: **{passed}** | needs review: **{n-passed}**.', '',
    '## Case decisions', '',
    '| Row | Decision | Max R-hat | Min ESS | Divergences | At max tree depth |',
    '|---:|---|---:|---:|---:|---:|']
for _, r in dispositions.iterrows():
    md.append(f"| {int(r.simulation_row)} | {r.review_disposition} | {float(r.max_r_hat):.4g} | {float(r.min_effective_sample_size):.4g} | {int(r.divergences)} | {float(r.fraction_at_max_tree_depth):.1%} |")
md += ['', '## Interpretation', '',
    '- **Row 87483:** keep unresolved. Fixing duration did not restore chain mixing; every draw reached maximum tree depth. Do not treat its posterior intervals as reliable.',
    '- **Row 4923:** keep under review. Most parameters mix acceptably, but `log_noise` ESS is 53.4 (<100), and about 85% of draws reach maximum tree depth.',
    '- Zero divergences is not sufficient evidence of convergence. More draws alone are not the recommended next move.',
    '- Both fits condition on a fixed period. Row 87483 also fixes duration, so it has no duration posterior uncertainty.',
    '- These are conditional synthetic checks; truth-in-interval counts are not calibration and do not establish real-star validity.', '',
    '## Next decision', '',
    'Stop repeating this NUTS setup. Investigate/reparameterize the likelihood and prior geometry before another sampler run.', '']
(OUT/'day9_17_review_report.md').write_text('\n'.join(md), encoding='utf-8')
print(dispositions[['simulation_row','review_disposition','max_r_hat','min_effective_sample_size','fraction_at_max_tree_depth']].to_string(index=False))
print('Saved review artifacts:', OUT)
print(json.dumps(report, indent=2))